# Final Model: Twitter-RoBERTa
**Nova IMS - Text Mining 2025/2026 - Group 31**

This notebook reproduces our final submission. The champion model is the fine-tuned
Twitter-RoBERTa (`cardiffnlp/twitter-roberta-base-sentiment`), trained on the full
training set, which reached a Macro F1 of 0.8453 on the validation split and beat the
DistilBERT, FinBERT and DeBERTa-v3 baselines.

This notebook is fully self-contained: all configuration and helper functions are
defined inline below (no imports from `src/`). The full model comparison and the
experiment log live in `tm_tests_31.ipynb`.

## Table of Contents

- [Setup](#setup)
  - [Imports](#imports)
  - [Configuration](#configuration)
  - [Utility functions](#utility-functions)
  - [Evaluation functions](#evaluation-functions)
  - [Train/validation split](#trainvalidation-split)
  - [Transformer trainer functions](#transformer-trainer-functions)
- [Train the champion](#train-the-champion)
- [Predict the test set and save the submission](#predict-the-test-set-and-save-the-submission)

## Setup

### Imports

In [8]:
import os, sys, csv
from datetime import datetime
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Run from the project root so data/ and outputs/ paths resolve.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\MartaFeria\Documents\NovaIms\TextMining\Project\repo


### Configuration

In [9]:
# === Configuration ===
SEED = 42
VAL_SIZE = 0.20

# Paths
TRAIN_CSV_PATH = "data/train.csv"
TEST_CSV_PATH = "data/test.csv"
RESULTS_CSV_PATH = "outputs/results.csv"
OUTPUT_PRED_PATH = "outputs/pred_best.csv"

# Labels
NUM_LABELS = 3
LABEL_NAMES = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
LABEL2ID = {"Bearish": 0, "Bullish": 1, "Neutral": 2}
ID2LABEL = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

# Leaderboard CSV schema
RESULTS_HEADERS = [
    "timestamp", "owner", "model_name", "feature_description",
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "parameters", "f1_per_class", "notes",
]

# Champion encoder (HF hub id, tokenized-dataset cache, checkpoint dir).
# The full DistilBERT / FinBERT / DeBERTa comparison lives in tm_tests_31.ipynb;
# this notebook only reproduces the winner.
ROBERTA_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
ROBERTA_CACHE_DIR = "outputs/roberta_cache"
ROBERTA_CHECKPOINT_DIR = "outputs/roberta_checkpoints"

### Utility functions

In [10]:
# === Utility functions ===
def log_info(msg: str) -> None:
    print(f"[INFO] {msg}")

def log_success(msg: str) -> None:
    print(f"[SUCCESS] {msg}")

def log_warning(msg: str) -> None:
    print(f"[WARNING] {msg}")

### Evaluation functions

In [11]:
# === Evaluation functions ===
def compute_metrics(y_true, y_pred) -> dict:
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {
        name: float(per_class[i]) if i < len(per_class) else 0.0
        for i, name in LABEL_NAMES.items()
    }
    return {
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "f1_per_class": f1_per_class,
    }


def log_model_run(
    model_name: str,
    feature_desc: str,
    metrics: dict,
    params: str = "",
    owner: str = "",
    notes: str = "",
) -> None:
    """Logs a model run to results.csv idempotently - updates if the same key exists."""
    os.makedirs(os.path.dirname(RESULTS_CSV_PATH), exist_ok=True)

    new_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "owner": owner,
        "model_name": model_name,
        "feature_description": feature_desc,
        "accuracy": f"{metrics['accuracy']:.4f}",
        "precision_macro": f"{metrics['precision_macro']:.4f}",
        "recall_macro": f"{metrics['recall_macro']:.4f}",
        "f1_macro": f"{metrics['f1_macro']:.4f}",
        "parameters": params,
        "f1_per_class": str(metrics.get("f1_per_class", "")),
        "notes": notes,
    }

    def _is_match(row: dict) -> bool:
        return (
            row["model_name"] == model_name
            and row["feature_description"] == feature_desc
            and row["parameters"] == params
            and row["owner"] == owner
        )

    existing_rows = []
    updated = False

    if os.path.exists(RESULTS_CSV_PATH) and os.path.getsize(RESULTS_CSV_PATH) > 0:
        try:
            with open(RESULTS_CSV_PATH, mode="r", newline="", encoding="utf-8") as f:
                reader = csv.reader(f)
                next(reader, None)
                for r in reader:
                    if len(r) != len(RESULTS_HEADERS):
                        continue
                    row = dict(zip(RESULTS_HEADERS, r))
                    if _is_match(row):
                        existing_rows.append(new_row)
                        updated = True
                    else:
                        existing_rows.append(row)
        except Exception as e:
            log_warning(f"Error reading leaderboard CSV: {e}. Resetting.")
            existing_rows = []

    if not updated:
        existing_rows.append(new_row)

    with open(RESULTS_CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_HEADERS)
        writer.writeheader()
        writer.writerows(existing_rows)

    log_info(f"{'Updated' if updated else 'Added'} run in {RESULTS_CSV_PATH}")


def save_submission(
    test_df: pd.DataFrame,
    predictions,
    output_path: str = OUTPUT_PRED_PATH,
    id_col: str = "id",
) -> pd.DataFrame:
    """Saves id + label predictions to CSV."""
    submission = pd.DataFrame({"id": test_df[id_col], "label": predictions})
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    submission.to_csv(output_path, index=False)
    log_success(f"Predictions saved to {output_path} ({len(submission)} rows)")
    return submission

### Train/validation split

In [12]:
# === Train/validation split ===
def stratified_split(
    dataset: pd.DataFrame,
    test_size: float = VAL_SIZE,
    seed: int = SEED,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
    X, y = dataset['text'], dataset['label']
    return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

### Transformer trainer functions

In [13]:
# === Transformer trainer functions ===
MAX_LENGTH = 128


@dataclass(frozen=True)
class TrainerSpec:
    """Everything that distinguishes one encoder trainer from another."""

    display_name: str    # label used in the results leaderboard
    model_name: str      # HF hub id
    cache_dir: str       # tokenized-dataset cache root
    checkpoint_dir: str  # Trainer output root


# Only the champion is reproduced here; the registry kept the same shape as
# tm_tests_31.ipynb so the helpers below are unchanged.
SPECS: dict[str, TrainerSpec] = {
    "roberta": TrainerSpec("Twitter-RoBERTa", ROBERTA_MODEL_NAME,
                           ROBERTA_CACHE_DIR, ROBERTA_CHECKPOINT_DIR),
}


def load_tokenizer(spec: TrainerSpec) -> AutoTokenizer:
    log_info(f"Loading tokenizer: {spec.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(spec.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def _tokenize_dataset(tokenizer, texts, labels=None, max_length: int = MAX_LENGTH) -> Dataset:
    """Build a tokenized HF Dataset from texts (and optional labels)."""
    data = {"text": list(texts)}
    if labels is not None:
        data["label"] = list(labels)
    return Dataset.from_dict(data).map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=max_length),
        batched=True,
        remove_columns=["text"],
    )


def build_hf_datasets(tokenizer, X_train, X_val, y_train, y_val, cache_dir: Path,
                      max_length: int = MAX_LENGTH):
    cache_dir.mkdir(parents=True, exist_ok=True)

    def _build(texts, labels, cache_path):
        if cache_path.exists():
            log_info(f"Loading dataset from cache: {cache_path}")
            return Dataset.load_from_disk(str(cache_path))
        ds = _tokenize_dataset(tokenizer, texts, labels, max_length=max_length)
        ds.save_to_disk(str(cache_path))
        log_info(f"Dataset cached to: {cache_path}")
        return ds

    return (_build(X_train, y_train, cache_dir / "train"),
            _build(X_val,   y_val,   cache_dir / "val"))


def make_compute_metrics(spec: TrainerSpec, notes: str = "", params: str = ""):
    def _compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        metrics = compute_metrics(labels, preds)
        log_model_run(
            model_name=spec.display_name,
            feature_desc="HF fine-tune",
            metrics=metrics,
            params=params or f"model={spec.model_name}, max_length={MAX_LENGTH}",
            notes=notes,
        )
        return {"accuracy": metrics["accuracy"], "f1_macro": metrics["f1_macro"]}
    return _compute_metrics


def build_trainer(spec: TrainerSpec, model, tokenizer, train_ds, val_ds, notes: str = "",
                  learning_rate: float = 2e-5, batch_size: int = 16,
                  seed: int = SEED, params: str = "") -> Trainer:
    training_args = TrainingArguments(
        output_dir=str(Path(spec.checkpoint_dir) / spec.model_name.replace("/", "_")),
        num_train_epochs=3,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=seed,
        logging_steps=50,
        report_to="none",
    )
    return Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=make_compute_metrics(spec, notes=notes, params=params),
    )


def run_trainer(model: str, n_samples: int | None = 500, notes: str = "",
                learning_rate: float = 2e-5, max_length: int = MAX_LENGTH,
                seed: int = SEED, batch_size: int = 16) -> Trainer:
    """Fine-tune the registered champion encoder.

    `model` is a key of SPECS; this notebook registers only "roberta".
    """
    spec = SPECS[model]

    log_info(f"Loading {'all' if n_samples is None else n_samples} samples ...")
    df = pd.read_csv(TRAIN_CSV_PATH)
    if n_samples:
        df = df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    X_train, X_val, y_train, y_val = stratified_split(df)
    log_info(f"Split - train={len(X_train)}, val={len(X_val)}")

    tokenizer = load_tokenizer(spec)

    # Sample-size-specific cache dir so a 500-sample spike isn't served when we
    # ask for the full dataset. max_length variants get their own cache too.
    suffix = "full" if n_samples is None else f"n{n_samples}"
    if max_length != MAX_LENGTH:
        suffix += f"_len{max_length}"
    cache_dir = Path(spec.cache_dir) / suffix
    train_ds, val_ds = build_hf_datasets(tokenizer, X_train, X_val, y_train, y_val, cache_dir,
                                         max_length=max_length)
    log_info(f"Datasets - train={len(train_ds)} rows, val={len(val_ds)} rows")

    log_info(f"Loading {spec.model_name} sequence classifier ...")
    # ignore_mismatched_sizes=True safely re-initializes the classification head
    # for our project's 3-label schema.
    hf_model = AutoModelForSequenceClassification.from_pretrained(
        spec.model_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    params = (f"model={spec.model_name}, max_length={max_length}, lr={learning_rate}, "
              f"seed={seed}, n={'full' if n_samples is None else n_samples}")
    trainer = build_trainer(spec, hf_model, tokenizer, train_ds, val_ds, notes=notes,
                            learning_rate=learning_rate, batch_size=batch_size,
                            seed=seed, params=params)

    log_info("Starting training ...")
    trainer.train()
    log_info("Running final evaluation ...")
    trainer.evaluate()
    log_success("Training complete.")
    return trainer


def predict_test_set(trainer, tokenizer, test_csv_path: str = TEST_CSV_PATH) -> np.ndarray:
    log_info(f"Loading test set from {test_csv_path}")
    test_df = pd.read_csv(test_csv_path)
    log_info(f"Test rows: {len(test_df)}")

    test_ds = _tokenize_dataset(tokenizer, test_df["text"])
    preds = trainer.predict(test_ds)
    labels = np.argmax(preds.predictions, axis=-1)
    log_success(f"Generated {len(labels)} test predictions.")
    return labels

## Train the champion

`run_trainer("roberta", n_samples=None, ...)` fine-tunes Twitter-RoBERTa on the full
training set and logs the run to `outputs/results.csv`.

In [14]:
trainer = run_trainer("roberta", n_samples=None, notes="final champion RoBERTa full data")

[INFO] Loading all samples ...
[INFO] Split - train=7634, val=1909
[INFO] Loading tokenizer: cardiffnlp/twitter-roberta-base-sentiment
[INFO] Loading dataset from cache: outputs\roberta_cache\full\train
[INFO] Loading dataset from cache: outputs\roberta_cache\full\val
[INFO] Datasets - train=7634 rows, val=1909 rows
[INFO] Loading cardiffnlp/twitter-roberta-base-sentiment sequence classifier ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[INFO] Starting training ...


c:\Users\MartaFeria\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

[INFO] Running final evaluation ...


c:\Users\MartaFeria\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[INFO] Updated run in outputs/results.csv


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.141211,0.350021,3,0.879518,0.846629


[SUCCESS] Training complete.


## Predict the test set and save the submission

We load the RoBERTa tokenizer through the same `load_tokenizer(spec)` helper used during
training, predict the held-out test set, and write `outputs/pred_31.csv` (id + predicted label).

In [15]:
tokenizer = load_tokenizer(SPECS["roberta"])
test_preds = predict_test_set(trainer, tokenizer, TEST_CSV_PATH)

test_df = pd.read_csv(TEST_CSV_PATH)
submission = save_submission(test_df, test_preds, output_path="outputs/pred_31.csv")
submission.head()

[INFO] Loading tokenizer: cardiffnlp/twitter-roberta-base-sentiment
[INFO] Loading test set from data/test.csv
[INFO] Test rows: 2388


Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

c:\Users\MartaFeria\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[SUCCESS] Generated 2388 test predictions.
[SUCCESS] Predictions saved to outputs/pred_31.csv (2388 rows)


,id,label
0,0,1
1,1,2
2,2,2
3,3,1
4,4,2
